<a href="https://colab.research.google.com/github/GalJakob/NLP/blob/main/post_asr_20250922_15_25.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [110]:


INPUT_COL   = "asr_output"
TARGET_COL  = "sentence"

DATASET_DIR1 = "1b10K_imvladikon-wav2vec2-xls-r-1b-hebrew"
DATASET_DIR2 = "300m10K_imvladikon-wav2vec2-xls-r-300m-hebrew"
DATASET_DIR3 = "53-hebrew"
MODEL_NAME_100K  = "Gal-Jakob/byt5small-h200-full" # trained on 100k real samples
MODEL_NAME_100k_PLUS_500k  = "Gal-Jakob/Post_asr_byt5small-h200-full-fixed256_full_ds" # trained on 100k + 500k synthetic

MAX_INPUT_LEN=256
MAX_TARGET_LEN=256


In [111]:
### THIS CELL is for checking and filtering ds ###

import torch
from datasets import load_from_disk

    
def _row_ok(x):
    src = (x.get(INPUT_COL) or "").strip()
    tgt = (x.get(TARGET_COL) or "").strip()
    return len(src) > 0 and len(tgt) > 0


def filter_dataset_by_bytes(ds,ds_name):
 
    def _len_bytes(s):
        return len((s or "").encode("utf-8"))

    def _keep_example(ex):
        return (
            _len_bytes(ex[INPUT_COL]) < MAX_INPUT_LEN
            and _len_bytes(ex[TARGET_COL]) < MAX_TARGET_LEN
        )

    before = len(ds)
    filtered = ds.filter(_keep_example)
    after = len(filtered)

    removed = before - after
    pct_removed = (removed / before * 100.0) if before > 0 else 0.0
    print(f"ds_name:  {ds_name}")
    print(f"Before:  {before}")
    print(f"After :  {after}")
    print(f"Removed: {removed} ({pct_removed:.2f}% removed)")
    print(f"-----")

    return filtered


ds_1b10k = load_from_disk(DATASET_DIR1)
ds_300m10k = load_from_disk(DATASET_DIR2)
ds_53_hebrew = load_from_disk(DATASET_DIR3)

ds_53_hebrew = ds_53_hebrew.rename_column("clean_sentence", "sentence")
ds_53_hebrew = ds_53_hebrew.rename_column("dirty_sentence", "asr_output")

ds_1b10k = ds_1b10k.filter(_row_ok)
ds_300m10k = ds_300m10k.filter(_row_ok)
ds_53_hebrew = ds_53_hebrew.filter(_row_ok)


print(f"any input/output above {MAX_INPUT_LEN} bytes will be Removed")
ds_1b10k = filter_dataset_by_bytes(ds_1b10k,"ds_1b10k")
ds_300m10k= filter_dataset_by_bytes(ds_300m10k,"ds_300m10k")
ds_53_hebrew= filter_dataset_by_bytes(ds_53_hebrew,"ds_53_hebrew")




any input/output above 256 bytes will be Removed
ds_name:  ds_1b10k
Before:  9997
After :  9376
Removed: 621 (6.21% removed)
-----
ds_name:  ds_300m10k
Before:  9993
After :  9471
Removed: 522 (5.22% removed)
-----
ds_name:  ds_53_hebrew
Before:  9995
After :  9633
Removed: 362 (3.62% removed)
-----


In [118]:
#base line 

from jiwer import wer, cer
import pandas as pd
from datasets import Dataset

def evaluate_text_only(ds, hyp_col="asr_output", ref_col="sentence"):  
    refs = [str(x) for x in ds[ref_col]]
    hyps = [str(x) for x in ds[hyp_col]] 
    metrics = {
        "samples": len(refs),
        "wer": float(wer(refs, hyps)),  # no text normalization
        "cer": float(cer(refs, hyps)),  # no text normalization
        "loss": None,  # true model loss (e.g., CTC) is not computable without logits/audio
    }
    print(metrics)
    return metrics


evaluate_text_only(ds_1b10k)  # ds_1b10k must have 'asr_output' and 'sentence' columns


{'samples': 9376, 'wer': 0.6557956271028316, 'cer': 0.2810548867781747, 'loss': None}


{'samples': 9376,
 'wer': 0.6557956271028316,
 'cer': 0.2810548867781747,
 'loss': None}

In [113]:
# --- Pre-tokenize the whole dataset once (keeps original text columns) ---

from transformers import AutoTokenizer

def tokenize_text2text_dataset(
    model_id_or_path,
    ds,
    input_col=INPUT_COL,
    ref_col=TARGET_COL,
    max_input_length=256,
    target_max_length=256,
):
    tokenizer = AutoTokenizer.from_pretrained(model_id_or_path, use_fast=False)

    def _preprocess(batch):
        srcs = [f"fix mistakes: {x}" for x in batch[INPUT_COL]]
        tgts = [x for x in batch[TARGET_COL]]
    
        # 1) Encode inputs
        model_inputs = tokenizer(
            srcs,
            truncation=True,
            max_length=MAX_INPUT_LEN,            
            padding="max_length"
        )
    
        # 2) Encode targets 
        target_enc = tokenizer(
            text_target=tgts,
            truncation=True,
            max_length=MAX_TARGET_LEN,
            padding="max_length"
        )
        model_inputs["labels"] = target_enc["input_ids"]
    
        return model_inputs

    tokenized = ds.map(_preprocess, batched=True)
    return tokenizer, tokenized


In [114]:

MODEL_ID = MODEL_NAME_100k_PLUS_500k  # or a local checkpoint path
MAX_INPUT_LEN = 256
MAX_TARGET_LEN = 256
BATCH_SIZE = 32

ds_eval = ds_53_hebrew

tokenizer, tokenized_ds = tokenize_text2text_dataset(
    model_id_or_path=MODEL_ID,
    ds=ds_eval,
    input_col=INPUT_COL,
    ref_col=TARGET_COL,
    max_input_length=MAX_INPUT_LEN,
    target_max_length=MAX_TARGET_LEN,
)


In [115]:
# --- Evaluate pre-tokenized seq2seq dataset (simple) ---
# Uses: tokenizer, tokenized_ds, MODEL_ID, INPUT_COL, TARGET_COL
# Computes: WER, CER, token-weighted cross-entropy LOSS

import torch
from torch.utils.data import DataLoader
from transformers import AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq
from jiwer import wer, cer

# Defaults if not already defined
BATCH_SIZE = globals().get("BATCH_SIZE", 32)
GEN_MAX_NEW_TOKENS = globals().get("GEN_MAX_NEW_TOKENS", 256)
NUM_BEAMS = globals().get("NUM_BEAMS", 1)
DO_SAMPLE = globals().get("DO_SAMPLE", True)

device = "cuda" if torch.cuda.is_available() else "cpu"

def infer_new(prompt, max_length: int = 128):
    input = tokenizer(f"fix mistakes: {prompt}", return_tensors="pt")
    input_ids      = input["input_ids"]
    attention_mask = input["attention_mask"]

    max_length = len(input_ids[0])
    # min_length = int(0.9 * max_length)
    print(max_length)
    output = model.generate(input_ids.to(device),
                            attention_mask=attention_mask.to(device),
                            max_new_tokens=max_length,
                            # min_length=min_length,
                            do_sample = True, top_k = 50, top_p = 0.85)
    output = tokenizer.decode(output[0], skip_special_tokens=True)
    return output

def _clip(s, n=256):
    s = str(s or "")
    return s if len(s) <= n else s[:n] + " ..."

    
# 1) Load model
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID).to(device).eval()

# 2) Keep refs/inputs for metrics (before stripping columns)
refs_all   = [str(x) for x in tokenized_ds[TARGET_COL]]
inputs_all = [str(x) for x in tokenized_ds[INPUT_COL]]
n = len(refs_all)

# 3) Build a clean dataset for the loader (only model columns)
keep_cols = [c for c in ["input_ids", "attention_mask", "labels"] if c in tokenized_ds.column_names]
dl_ds = tokenized_ds.remove_columns([c for c in tokenized_ds.column_names if c not in keep_cols])

# Let the collator pad & convert to tensors; mask labels padding with -100
collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, label_pad_token_id=-100)
loader = DataLoader(dl_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collator)

# 4) Loop: compute loss and generate preds
preds_all = []
total_loss_sum = 0.0   # sum of (batch_loss * valid_target_tokens)
total_tokens   = 0     # count of valid (non -100) target tokens
# Track progress during evaluation with a tqdm progress bar

from tqdm.auto import tqdm

preds_all = []
total_loss_sum = 0.0   # sum of (batch_loss * valid_target_tokens)
total_tokens   = 0     # count of valid (non -100) target tokens

pbar = tqdm(total=len(loader), desc="Evaluating", unit="batch")

for step, batch in enumerate(loader, 1):
    processed_before = len(preds_all)
    # Tensors (DataCollatorForSeq2Seq already tensorized them)
    labels = batch.pop("labels").to(device)
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch.get("attention_mask")
    attention_mask = attention_mask.to(device) if attention_mask is not None else None

    with torch.inference_mode():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

        valid_tokens = (labels != -100).sum().item()
        total_loss_sum += loss.item() * max(valid_tokens, 1)
        total_tokens   += max(valid_tokens, 1)
        gen = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=GEN_MAX_NEW_TOKENS,
            num_beams=NUM_BEAMS,
            do_sample = True, top_k = 50, top_p = 0.85
        )
        preds = tokenizer.batch_decode(gen, skip_special_tokens=True)

    preds_all.extend(preds)
    
    # Update progress bar with running stats
    processed = len(preds_all)           # robust even for last partial batch
    avg_loss = total_loss_sum / max(total_tokens, 1)
    pbar.set_postfix_str(f"samples={processed}/{n}, avg_loss={avg_loss:.4f}")
    pbar.update(1)
    
    if step %100 == 0:
        k = min(2, len(preds))
        local_idxs = list(range(k))  # or use random.sample(range(len(preds)), k)
        print("\n" + "="*72)
        print(f"Preview @ batch {step} (showing {k} examples):")
        for li in local_idxs:
            gi = processed_before + li  # global index in the dataset
            print("-"*72)
            print(f"[global {gi}]")
            print(f"IN : {_clip(inputs_all[gi])}")
            print(f"REF: {_clip(refs_all[gi])}")
            print(f"PRED:{_clip(preds[li])}")
        print("="*72 + "\n")

pbar.close()

# Final metrics (no normalization)
results = {
    "samples": n,
    "wer": float(wer(refs_all, preds_all)),
    "cer": float(cer(refs_all, preds_all)),
    "loss": float(total_loss_sum / total_tokens) if total_tokens > 0 else None,
}
print("Results:", results)



Evaluating:   0%|          | 0/302 [00:00<?, ?batch/s]


Preview @ batch 100 (showing 2 examples):
------------------------------------------------------------------------
[global 3168]
IN : נכון מבחינתי שברגע שאחרי שהתלתי ל עסוק יבנושא של מפרסיות שמלנעות על ידי
REF: נכון מבחינתי שברגע שאחרי שהתחלתי לעסוק בנושא של מפרסיות שמונעות על ידי
PRED:נכון מבחינתי שברגע שאחרי שהתללתי לי עסוק בנושא של מפרסיות שמלנעות על ידי
------------------------------------------------------------------------
[global 3169]
IN : הדמיון תשלי התפתח
REF: הדמיון שלי התפתח
PRED:הדמיון תשלי התפתח


Preview @ batch 200 (showing 2 examples):
------------------------------------------------------------------------
[global 6368]
IN : זה הידשטםנדים משתמשים בזה כרוצים
REF: זה הידשטרנדים משתמשים בזה כרוצים
PRED:זה הידשטנדים משתמשים בזה כרוצים
------------------------------------------------------------------------
[global 6369]
IN : שי אךכיטקט של להמערכת הבאה שלה המודלי שלנו
REF: שיפ ארכיטקט של המערכת הבאה שלהם המודלים שלנו
PRED:שי אך כיטקט של להמערכת הבאה שלה המודלי שלנו


Prev

In [ ]:
print(results)
print(preds)
print(preview)
